# Battery Health — Dataset Preparation

Produces two analysis-ready CSV files:

| File | Batteries | Resistance | Use |
|---|---|---|---|
| `data/processed/controlled.csv` | Room-temp (B0005–B0018) | EIS `Re`/`Rct` + onset proxy `r_proxy` | Upper-bound model (lab conditions) |
| `data/processed/randomized.csv` | RW1–RW20 (room temp) | Onset proxy `r_proxy` only | Field-realistic model |

**Design:** EIS impedance is the gold-standard resistance signal but requires batteries to be at rest — unavailable in real renewable-energy installations. The onset DCIR proxy (`r_proxy = ΔV/ΔI` at reference-discharge onset) is computable from ordinary telemetry and validated against EIS in `01_eda.ipynb` (r ≈ 0.77 on controlled, r ≈ 0.79 on RW). The controlled CSV retains EIS to enable an upper-bound model; the randomized CSV uses only the proxy — the realistic field scenario.

## 1. Imports & paths

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src.battery import (
    load_controlled, cycles_to_df,
    load_randomized, rw_steps_to_df,
    load_impedance, attach_impedance,
    compute_rul, engineer_features,
)

warnings.filterwarnings("ignore")

DATA    = Path("../data")
CTRL    = DATA / "controlled"
RW      = DATA / "randomized"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(exist_ok=True)


## 2. Battery registries

**Excluded:** hot (B0029–B0032) and cold (B0042–B0044) batteries.  
Hot batteries never reach the 80 % EOL threshold (≤6 % fade in 40 cycles). Cold group is internally inconsistent (B0042 → 0.889 Ahr vs B0043/44 ~1.49 Ahr despite the same temperature condition). Both retained in `01_eda.ipynb` for temperature-effect plots; excluded here to avoid poisoning the RUL labels.
RW2 and RW18 were dropped as well as their sensor readings are all -4093 degrees.

In [2]:
G1 = CTRL / "1. BatteryAgingARC-FY08Q4"
G3 = CTRL / "3. BatteryAgingARC_25-44"

CONTROLLED_BATTERIES = [
    ("B0005", 2.7, "room", G1 / "B0005.mat"),
    ("B0006", 2.5, "room", G1 / "B0006.mat"),
    ("B0007", 2.2, "room", G1 / "B0007.mat"),
    ("B0018", 2.5, "room", G1 / "B0018.mat"),
]

RW_UNI_VAR  = RW / "Battery_Uniform_Distribution_Variable_Charge_Room_Temp_DataSet_2Post" / "data" / "Matlab"
RW_UNI_CD   = RW / "Battery_Uniform_Distribution_Charge_Discharge_DataSet_2Post"          / "data" / "Matlab"
RW_SKW_LOW  = RW / "RW_Skewed_Low_Room_Temp_DataSet_2Post"                                / "data" / "Matlab"
RW_SKW_HIGH = RW / "RW_Skewed_High_Room_Temp_DataSet_2Post"                               / "data" / "Matlab"

RW_BATTERIES = [
    ("RW1",  "uniform_var_charge", RW_UNI_VAR  / "RW1.mat"),
    ("RW9",  "uniform_cd",         RW_UNI_CD   / "RW9.mat"),
    ("RW13", "skewed_low",         RW_SKW_LOW  / "RW13.mat"),
    ("RW14", "skewed_low",         RW_SKW_LOW  / "RW14.mat"),
    ("RW15", "skewed_low",         RW_SKW_LOW  / "RW15.mat"),
    ("RW16", "skewed_low",         RW_SKW_LOW  / "RW16.mat"),
    ("RW17", "skewed_high",        RW_SKW_HIGH / "RW17.mat"),
    ("RW19", "skewed_high",        RW_SKW_HIGH / "RW19.mat"),
    ("RW20", "skewed_high",        RW_SKW_HIGH / "RW20.mat"),
]


## 3. Load raw cycles (r_proxy already embedded in loaders)

In [3]:
ctrl_dfs = []
for battery_id, cutoff_v, temp_cond, path in CONTROLLED_BATTERIES:
    df = cycles_to_df(load_controlled(path), battery_id, cutoff_v, temp_cond)
    ctrl_dfs.append(df)
    print(f"{battery_id}: {len(df):3d} cycles | r_proxy {df.r_proxy.min():.3f}..{df.r_proxy.max():.3f}")

controlled_raw = pd.concat(ctrl_dfs, ignore_index=True)

rw_dfs = []
for battery_id, rw_cond, path in RW_BATTERIES:
    df = rw_steps_to_df(load_randomized(path), battery_id, rw_cond)
    if df.empty:
        print(f"{battery_id}: no reference steps found — skipped")
        continue
    rw_dfs.append(df)
    print(f"{battery_id} ({rw_cond}): {len(df):2d} ref cycles | r_proxy {df.r_proxy.min():.3f}..{df.r_proxy.max():.3f}")

randomized_raw = pd.concat(rw_dfs, ignore_index=True)


B0005: 168 cycles | r_proxy 0.130..0.176
B0006: 168 cycles | r_proxy 0.135..0.208
B0007: 168 cycles | r_proxy 0.129..0.165
B0018: 132 cycles | r_proxy 0.138..0.181
RW1 (uniform_var_charge): 24 ref cycles | r_proxy 0.179..0.262
RW9 (uniform_cd): 40 ref cycles | r_proxy 0.141..0.300
RW13 (skewed_low): 13 ref cycles | r_proxy 0.165..0.338
RW14 (skewed_low): 13 ref cycles | r_proxy 0.170..0.402
RW15 (skewed_low): 14 ref cycles | r_proxy 0.170..0.389
RW16 (skewed_low): 11 ref cycles | r_proxy 0.168..0.377
RW17 (skewed_high): 17 ref cycles | r_proxy 0.167..0.343
RW19 (skewed_high): 16 ref cycles | r_proxy 0.165..0.341
RW20 (skewed_high): 17 ref cycles | r_proxy 0.168..0.353


## 4. EIS join — controlled only

**Why only controlled:** EIS requires a dedicated rest + frequency-sweep protocol not present in the randomized datasets. `Re` and `Rct` are joined by timestamp nearest-neighbour (tolerance 48 h). The first ~20 discharge cycles of each battery pre-date the first EIS sweep; those rows are back-filled from the earliest available measurement — a conservative choice that slightly underestimates early-life resistance increase.

In [4]:
ctrl_joined_dfs = []
for battery_id, cutoff_v, temp_cond, path in CONTROLLED_BATTERIES:
    df   = controlled_raw[controlled_raw["battery_id"] == battery_id].copy()
    imp  = load_impedance(path)
    df   = attach_impedance(df, imp)
    nan_count = df[["Re","Rct"]].isna().any(axis=1).sum()
    print(f"{battery_id}: Re {df.Re.min():.4f}..{df.Re.max():.4f} | Rct {df.Rct.min():.4f}..{df.Rct.max():.4f} | NaN rows after bfill: {nan_count}")
    ctrl_joined_dfs.append(df)

controlled_raw = pd.concat(ctrl_joined_dfs, ignore_index=True)


B0005: Re 0.0436..0.0631 | Rct 0.0677..0.0892 | NaN rows after bfill: 0
B0006: Re 0.0591..0.0791 | Rct 0.0780..0.1067 | NaN rows after bfill: 0
B0007: Re 0.0359..0.0673 | Rct 0.0601..0.0936 | NaN rows after bfill: 0
B0018: Re 0.0602..0.0661 | Rct 0.0841..0.0957 | NaN rows after bfill: 0


## 5. Physically grounded RUL

**EOL threshold: 80 % capacity retention (20 % fade).**  
Chosen because: (a) it is the battery-industry standard for "retired" cells, (b) every battery in the model set crosses it (unlike the 70 % NASA spec for room temp).

`rul_cycles` = remaining cycles to first sub-80 % crossing (clipped to 0 after EOL).  
`rul_frac`   = `rul_cycles / eol_index` ∈ [0, 1] (normalised).  
`rul_censored` = True for batteries that never reach EOL in the dataset (none expected in this model set, but flagged for safety).

In [5]:
EOL_THRESHOLD = 0.80

# Apply per battery (RUL is relative to each battery's own baseline)
def apply_rul(df):
    parts = []
    for bid, grp in df.groupby("battery_id"):
        parts.append(compute_rul(grp.copy(), threshold=EOL_THRESHOLD))
    return pd.concat(parts, ignore_index=True)

controlled_raw = apply_rul(controlled_raw)
randomized_raw = apply_rul(randomized_raw)

print("Controlled RUL summary (first/last cycle per battery):")
for bid, g in controlled_raw.groupby("battery_id"):
    eol = g["eol_index"].iloc[0]; cens = g["rul_censored"].iloc[0]
    print(f"  {bid}: EOL@cycle {eol}  censored={cens}  rul_cycles[0]={g.rul_cycles.iloc[0]}")

print("\nRandomized RUL summary:")
for bid, g in randomized_raw.groupby("battery_id"):
    eol = g["eol_index"].iloc[0]; cens = g["rul_censored"].iloc[0]
    print(f"  {bid}: EOL@cycle {eol}  censored={cens}")


Controlled RUL summary (first/last cycle per battery):
  B0005: EOL@cycle 99  censored=False  rul_cycles[0]=99
  B0006: EOL@cycle 60  censored=False  rul_cycles[0]=60
  B0007: EOL@cycle 123  censored=False  rul_cycles[0]=123
  B0018: EOL@cycle 76  censored=False  rul_cycles[0]=76

Randomized RUL summary:
  RW1: EOL@cycle 11  censored=False
  RW13: EOL@cycle 5  censored=False
  RW14: EOL@cycle 4  censored=False
  RW15: EOL@cycle 4  censored=False
  RW16: EOL@cycle 4  censored=False
  RW17: EOL@cycle 6  censored=False
  RW19: EOL@cycle 6  censored=False
  RW20: EOL@cycle 7  censored=False
  RW9: EOL@cycle 7  censored=False


## 6. Feature engineering

All features are **causal** (computed from current and past cycles only — no future leakage). Rolling statistics use a window of 5 cycles with `min_periods=1`.

| Feature | Description |
|---|---|
| `*_roll_mean / *_roll_std` | Rolling mean/std of `voltage_mean`, `temp_mean`, `r_proxy` |
| `cap_fade_rate` | Capacity change per cycle (rolling-smoothed diff) |
| `r_proxy_rate` | Rate of change of resistance proxy |
| `cycle_norm` | Normalised cycle position within observed battery life |
| `cap_delta_from_baseline` | Capacity relative to battery-specific initial value |

**Excluded from RW features:** `voltage_min` (constant 3.2 V floor, no signal).

In [6]:
def apply_features(df):
    parts = []
    for bid, grp in df.groupby("battery_id"):
        parts.append(engineer_features(grp.copy()))
    return pd.concat(parts, ignore_index=True)

controlled_raw = apply_features(controlled_raw)
randomized_raw = apply_features(randomized_raw)

print("Controlled feature columns:", [c for c in controlled_raw.columns if any(
    x in c for x in ["roll","rate","norm","delta"])])
print("Randomized feature columns:", [c for c in randomized_raw.columns if any(
    x in c for x in ["roll","rate","norm","delta"])])


Controlled feature columns: ['voltage_mean_5_roll_mean', 'voltage_mean_5_roll_std', 'voltage_mean_10_roll_mean', 'voltage_mean_10_roll_std', 'voltage_mean_15_roll_mean', 'voltage_mean_15_roll_std', 'voltage_mean_20_roll_mean', 'voltage_mean_20_roll_std', 'temp_mean_5_roll_mean', 'temp_mean_5_roll_std', 'temp_mean_10_roll_mean', 'temp_mean_10_roll_std', 'temp_mean_15_roll_mean', 'temp_mean_15_roll_std', 'temp_mean_20_roll_mean', 'temp_mean_20_roll_std', 'r_proxy_5_roll_mean', 'r_proxy_5_roll_std', 'r_proxy_10_roll_mean', 'r_proxy_10_roll_std', 'r_proxy_15_roll_mean', 'r_proxy_15_roll_std', 'r_proxy_20_roll_mean', 'r_proxy_20_roll_std', 'cap_fade_rate_5', 'cap_fade_rate_10', 'cap_fade_rate_15', 'cap_fade_rate_20', 'r_proxy_rate_5', 'r_proxy_rate_10', 'r_proxy_rate_15', 'r_proxy_rate_20', 'cycle_norm', 'cap_delta_from_baseline']
Randomized feature columns: ['voltage_mean_5_roll_mean', 'voltage_mean_5_roll_std', 'voltage_mean_10_roll_mean', 'voltage_mean_10_roll_std', 'voltage_mean_15_roll

## 7. Train / test split

**Strategy: hold out whole batteries (~1/3 per regime), `random_state=42`.**  
Rationale: splitting by time within a battery leaks early-life context to the test set. Holding out entire batteries ensures the model has never seen the test battery's degradation history — the realistic deployment scenario (predict RUL of a new battery from telemetry).

In [7]:
import random

def battery_split(df, id_col, test_frac=0.33, random_state=42):
    rng = random.Random(random_state)
    ids = sorted(df[id_col].unique().tolist())
    rng.shuffle(ids)
    n_test  = max(1, round(len(ids) * test_frac))
    test_ids = set(ids[:n_test])
    df = df.copy()
    df["split"] = df[id_col].apply(lambda x: "test" if x in test_ids else "train")
    return df

controlled_raw = battery_split(controlled_raw, "battery_id")
randomized_raw = battery_split(randomized_raw, "battery_id")

print("Controlled split:")
for s, g in controlled_raw.groupby("split"):
    print(f"  {s}: {g.battery_id.unique().tolist()}")

print("\nRandomized split:")
for s, g in randomized_raw.groupby("split"):
    print(f"  {s}: {g.battery_id.unique().tolist()}")


Controlled split:
  test: ['B0007']
  train: ['B0005', 'B0006', 'B0018']

Randomized split:
  test: ['RW15', 'RW19', 'RW20']
  train: ['RW1', 'RW13', 'RW14', 'RW16', 'RW17', 'RW9']


## 8. Export

Two CSVs with aligned column schemas. The only asymmetry is `Re`/`Rct` (EIS, controlled only) and `voltage_min` (meaningful only for controlled — excluded from randomized). Both CSVs include `battery_id`, `cycle_index`, `split`, both targets (`rul_cycles`, `rul_frac`), `capacity_retention`, and `energy_retention`.

**`energy_wh` / `energy_retention`** — discharge energy via V·|I| integration, normalised to the battery's own early-life baseline. Used as the *efficiency* target in `03_train.ipynb`. Round-trip efficiency (E_discharge / E_charge) was the original candidate but is ill-defined on random-walk batteries (~40% of RW reference cycles yield RTE > 1 due to partial charges). Discharge-side energy retention is robustly computable on both datasets.

**Training notebook (`03_train.ipynb`) usage:**
- Phase 1 — Oracle: `controlled.csv`, all features incl. EIS `Re`/`Rct`.
- Phase 2 — Practical: merge both CSVs on shared columns, `r_proxy` only.
- Lifespan target: `rul_frac`; Efficiency target: `energy_retention`.


In [8]:
CTRL_KEEP = [
    "battery_id", "cycle_index", "split",
    "capacity_ahr", "capacity_retention",
    "energy_wh", "energy_retention",
    "voltage_min", "voltage_mean",
    "temp_mean", "temp_max",
    "duration_s", "avg_current_a",
    "r_proxy", "Re", "Rct",
    "voltage_mean_roll_mean", "voltage_mean_roll_std",
    "temp_mean_roll_mean",    "temp_mean_roll_std",
    "r_proxy_roll_mean",      "r_proxy_roll_std",
    "cap_fade_rate", "r_proxy_rate",
    "cycle_norm", "cap_delta_from_baseline",
    "rul_cycles", "rul_frac", "rul_censored", "eol_index",
]

RW_KEEP = [c for c in CTRL_KEEP if c not in ("voltage_min", "Re", "Rct", "avg_current_a")]
RW_KEEP = RW_KEEP + ["temp_sensor_ok"]

ctrl_out = controlled_raw[[c for c in CTRL_KEEP if c in controlled_raw.columns]]
rw_out   = randomized_raw[[c for c in RW_KEEP   if c in randomized_raw.columns]]

ctrl_path = PROCESSED / "controlled.csv"
rw_path   = PROCESSED / "randomized.csv"

ctrl_out.to_csv(ctrl_path, index=False)
rw_out.to_csv(rw_path,   index=False)

print(f"controlled.csv: {len(ctrl_out)} rows × {len(ctrl_out.columns)} cols  →  {ctrl_path}")
print(f"randomized.csv: {len(rw_out)}   rows × {len(rw_out.columns)}   cols  →  {rw_path}")
print(f"\nControlled NaN count:\n{ctrl_out.isna().sum()[ctrl_out.isna().sum()>0]}")
print(f"\nRandomized NaN count:\n{rw_out.isna().sum()[rw_out.isna().sum()>0]}")


controlled.csv: 636 rows × 22 cols  →  ../data/processed/controlled.csv
randomized.csv: 165   rows × 18   cols  →  ../data/processed/randomized.csv

Controlled NaN count:
Series([], dtype: int64)

Randomized NaN count:
Series([], dtype: int64)


## 9. Sanity checks

In [9]:
for name, df in [("controlled", ctrl_out), ("randomized", rw_out)]:
    assert (df["rul_cycles"] >= 0).all(), f"{name}: negative rul_cycles"
    for bid in df["battery_id"].unique():
        splits = df.loc[df["battery_id"]==bid, "split"].unique()
        assert len(splits)==1, f"{name}: {bid} appears in both train and test!"
    print(f"{name}: ✓  rul_cycles ≥ 0  |  no battery split across train/test")

# r_proxy trend: should increase (resistance grows with age) for most batteries
print("\nr_proxy first→last per battery (should generally increase):")
for name, df in [("ctrl", ctrl_out), ("rw", rw_out)]:
    for bid, g in df.groupby("battery_id"):
        g = g.sort_values("cycle_index")
        direction = "↑" if g.r_proxy.iloc[-1] > g.r_proxy.iloc[0] else "↓"
        print(f"  [{name}] {bid}: {g.r_proxy.iloc[0]:.3f} → {g.r_proxy.iloc[-1]:.3f} {direction}")


controlled: ✓  rul_cycles ≥ 0  |  no battery split across train/test
randomized: ✓  rul_cycles ≥ 0  |  no battery split across train/test

r_proxy first→last per battery (should generally increase):
  [ctrl] B0005: 0.165 → 0.168 ↑
  [ctrl] B0006: 0.158 → 0.199 ↑
  [ctrl] B0007: 0.163 → 0.158 ↓
  [ctrl] B0018: 0.147 → 0.179 ↑
  [rw] RW1: 0.235 → 0.262 ↑
  [rw] RW13: 0.165 → 0.338 ↑
  [rw] RW14: 0.170 → 0.402 ↑
  [rw] RW15: 0.170 → 0.389 ↑
  [rw] RW16: 0.168 → 0.377 ↑
  [rw] RW17: 0.174 → 0.343 ↑
  [rw] RW19: 0.173 → 0.341 ↑
  [rw] RW20: 0.177 → 0.353 ↑
  [rw] RW9: 0.141 → 0.300 ↑
